In [ ]:
!pip install datasets transformers -q
!pip install "unsloth[colab-new]" -q

In [ ]:
import unsloth
import transformers
import datasets
import trl

print(f"unsloth:        {unsloth.__version__}")
print(f"transformers:   {transformers.__version__}")
print(f"datasets:       {datasets.__version__}")
print(f"trl:            {trl.__version__}")

In [ ]:
from huggingface_hub import login
login('HT') 

In [ ]:
ex = ds[0]
print(f"Колонки: {ds.column_names}")
print(f"target = {ex['target']}")
print(f"nums   = {ex['nums']}")
print(f"messages: {len(ex['messages'])} сообщений")
for msg in ex['messages']:
    role = msg['role']
    content = msg['content']
    print(f"\n  [{role}]")
    print(f"  {content[:300]}")

In [ ]:
# Смотрим полный ответ ассистента
ex = ds[0]
asst = ex['messages'][2]  # третье сообщение
print(f"role: {asst['role']}")
print(f"Длина content: {len(asst['content'])} символов")
print()
print("--- Первые 500 символов ---")
print(asst['content'][:500])
print()
print("--- Последние 300 символов ---")
print(asst['content'][-300:])

In [ ]:
import re

# Проверяем первые 10 примеров
print("Проверка формата <answer> в первых 10 примерах:\n")
for i in range(10):
    ex = ds[i]
    asst_content = ex['messages'][2]['content']
    
    m = re.search(r'<answer>(.*?)</answer>', asst_content, re.DOTALL)
    if m:
        answer_raw = m.group(1).strip()
        print(f"[{i}] nums={ex['nums']}, target={ex['target']}")
        print(f"     answer: {repr(answer_raw)}")
        
        # Проверяем: есть ли '=' в ответе
        has_eq = '=' in answer_raw
        
        # Извлекаем только выражение (до '=')
        if has_eq:
            expr = answer_raw.split('=')[0].strip()
        else:
            expr = answer_raw
            
        # Валидируем
        try:
            result = eval(expr)
            correct = abs(result - ex['target']) < 1e-6
            print(f"     eval={result:.4f}, correct={correct}")
        except Exception as e:
            print(f"     eval ERROR: {e}")
    else:
        print(f"[{i}] НЕТ ТЕГА <answer>!")
    print()

In [ ]:
# Проверяем совместимость ключевых API
from transformers import AutoTokenizer
from datasets import load_dataset

# 1. Токенизатор Gemma
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Chat template: {'есть' if tokenizer.chat_template else 'НЕТ — проблема!'}")

# 2. Проверяем apply_chat_template
test_msgs = [
    {"role": "user", "content": "test"}
]
try:
    out = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
    print(f"apply_chat_template: OK")
    print(f"Пример: {repr(out[:80])}")
except Exception as e:
    print(f"apply_chat_template: ОШИБКА — {e}")

# 3. Загрузка датасета
ds = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "verified_Qwen2.5-7B-Instruct", split="train")
print(f"\nДатасет загружен: {len(ds)} примеров")
print(f"Колонки: {ds.column_names}")

# 4. Структура одного примера
ex = ds[0]
print(f"\nПример #0:")
print(f"  target = {ex['target']}")
print(f"  nums   = {ex['nums']}")
print(f"  prompt = список из {len(ex['messages'])} сообщений")
for msg in ex['messages']:
    print(f"    role={msg['role']}, len={len(msg['content'])}")

In [ ]:
import re
import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer
 
warnings.filterwarnings("ignore")

In [ ]:
# ── НАСТРОЙКИ ─────────────────────────────────────────────────────────
DATASET_NAME    = "HuggingFaceTB/Countdown-Task-GOLD"
VERIFIED_CONFIG = "verified_Qwen2.5-7B-Instruct"
ALL_CONFIG      = "all"
MODEL_NAME      = "google/gemma-3-1b-it"
MAX_SEQ_OPTIONS = [512, 768, 1024, 1536, 2048]
TOKENIZE_SAMPLE = 2000   # сколько примеров токенизировать (скорость vs точность)
VALIDATE_SAMPLE = 500    # сколько проверять математически

In [ ]:
print("=" * 70)
print("  СЕКЦИЯ 1: ЗАГРУЗКА ДАТАСЕТОВ")
print("=" * 70)
 
ds_verified = load_dataset(DATASET_NAME, VERIFIED_CONFIG, split="train")
ds_all      = load_dataset(DATASET_NAME, ALL_CONFIG,      split="train")
 
print(f"\nverified ({VERIFIED_CONFIG}): {len(ds_verified):>6} примеров")
print(f"all:                          {len(ds_all):>6} примеров")
print(f"Колонки: {ds_verified.column_names}")

# Показываем структуру одного примера
ex0 = ds_verified[0]
print(f"\n── Пример #0 ──────────────────────────────────────────────────")
print(f"  target  : {ex0['target']}")
print(f"  nums    : {ex0['nums']}")
print(f"  messages: {len(ex0['messages'])} сообщений")
for i, msg in enumerate(ex0['messages']):
    preview = msg['content'].replace('\n', ' ')[:100]
    print(f"    [{i}] role={msg['role']:9} | {preview}...")

In [ ]:
print("\n" + "=" * 70)
print("  СЕКЦИЯ 2: БАЗОВАЯ СТАТИСТИКА")
print("=" * 70)
 
def dataset_stats(ds, name):
    nums_lens = [len(ex['nums']) for ex in ds]
    targets   = [ex['target']   for ex in ds]
    cnt = Counter(nums_lens)
    total = len(ds)
 
    print(f"\n── {name} ({total} примеров) ──────────────────────────────────")
    print(f"  Количество чисел (nums):")
    for k in sorted(cnt):
        print(f"    {k} чисел: {cnt[k]:>6}  ({cnt[k]/total*100:5.1f}%)")
 
    print(f"\n  Target:  min={min(targets)}  max={max(targets)}"
          f"  mean={np.mean(targets):.0f}"
          f"  p50={np.percentile(targets,50):.0f}"
          f"  p90={np.percentile(targets,90):.0f}")
    return cnt, targets
 
cnt_verified, tgt_verified = dataset_stats(ds_verified, VERIFIED_CONFIG)
cnt_all,      tgt_all      = dataset_stats(ds_all,      "all")

In [ ]:
print("\n" + "=" * 70)
print("  СЕКЦИЯ 3: ТОКЕНИЗАЦИЯ")
print("=" * 70)
 
print(f"\nЗагружаем токенизатор {MODEL_NAME} ...")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"  vocab_size    = {tok.vocab_size:,}")
print(f"  chat_template = {'есть' if tok.chat_template else 'НЕТ!'}")
print(f"  bos_token     = {repr(tok.bos_token)}")
print(f"  eos_token     = {repr(tok.eos_token)}")
 
# Показываем как выглядит промпт после apply_chat_template
sample_user_msgs = [
    {"role": "system", "content": ex0['messages'][0]['content']},
    {"role": "user",   "content": ex0['messages'][1]['content']},
]
prompt_rendered = tok.apply_chat_template(
    sample_user_msgs, tokenize=False, add_generation_prompt=True
)
print(f"\n── Пример промпта после apply_chat_template ────────────────────")
print(prompt_rendered[:400])
print("...")
 
def tokenize_example(ex, tokenizer):
    """Возвращает (prompt_len, answer_len, total_len)."""
    msgs = ex['messages']
    # Промпт = system + user
    user_part = [m for m in msgs if m['role'] in ('system', 'user')]
    prompt_text = tokenizer.apply_chat_template(
        user_part, tokenize=False, add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
 
    # Ответ = content assistant
    asst = next((m for m in msgs if m['role'] == 'assistant'), None)
    if asst:
        asst_ids = tokenizer(asst['content'], add_special_tokens=False)['input_ids']
    else:
        asst_ids = []
 
    p = len(prompt_ids)
    a = len(asst_ids)
    return p, a, p + a
 
print(f"\nТокенизируем {TOKENIZE_SAMPLE} примеров из verified ...")
sample = ds_verified.select(range(min(TOKENIZE_SAMPLE, len(ds_verified))))
 
prompt_lens = []
answer_lens = []
total_lens  = []
nums_counts = []
 
for ex in sample:
    p, a, t = tokenize_example(ex, tok)
    prompt_lens.append(p)
    answer_lens.append(a)
    total_lens.append(t)
    nums_counts.append(len(ex['nums']))
 
prompt_lens = np.array(prompt_lens)
answer_lens = np.array(answer_lens)
total_lens  = np.array(total_lens)
nums_counts = np.array(nums_counts)
 
def print_stats(arr, label):
    print(f"\n── {label} ─────────────────────────────────────────────────")
    for name, val in [
        ("min",  arr.min()),  ("max",  arr.max()),
        ("mean", arr.mean()), ("std",  arr.std()),
        ("p50",  np.percentile(arr, 50)),
        ("p75",  np.percentile(arr, 75)),
        ("p90",  np.percentile(arr, 90)),
        ("p95",  np.percentile(arr, 95)),
        ("p99",  np.percentile(arr, 99)),
    ]:
        print(f"  {name:>5} = {val:>8.1f} токенов")
 
print_stats(prompt_lens, "ДЛИНА ПРОМПТА (system + user)")
print_stats(answer_lens, "ДЛИНА ОТВЕТА  (assistant)")
print_stats(total_lens,  "ПОЛНАЯ ДЛИНА  (prompt + answer)")
 
print(f"\n── Длина ответа по количеству чисел ────────────────────────────")
hdr = f"  {'nums':>5} | {'n':>5} | {'min':>5} | {'mean':>6} | {'p75':>6} | {'p90':>6} | {'p99':>6} | {'max':>6}"
print(hdr)
print("  " + "-" * (len(hdr) - 2))
for n in sorted(set(nums_counts)):
    mask = nums_counts == n
    a = answer_lens[mask]
    t = total_lens[mask]
    print(f"  {n:>5} | {mask.sum():>5} | {a.min():>5.0f} | {a.mean():>6.0f} | "
          f"{np.percentile(a,75):>6.0f} | {np.percentile(a,90):>6.0f} | "
          f"{np.percentile(a,99):>6.0f} | {a.max():>6.0f}")

In [ ]:
print("\n" + "=" * 70)
print("  СЕКЦИЯ 3: ТОКЕНИЗАЦИЯ — ДЛИНЫ ПРОМПТА И ОТВЕТА")
print("=" * 70)

from transformers import AutoTokenizer

print(f"\nЗагружаем токенизатор {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"  Vocab size       : {tokenizer.vocab_size:,}")
print(f"  Chat template    : {'да' if tokenizer.chat_template else 'нет'}")
print(f"  BOS token        : {repr(tokenizer.bos_token)}")
print(f"  EOS token        : {repr(tokenizer.eos_token)}")
print(f"  PAD token        : {repr(tokenizer.pad_token)}")

In [ ]:
def tokenize_example(ex, tokenizer):
    """
    Возвращает длину prompt-части и ответа по отдельности.
    """
    msgs = ex['prompt']

    # Разделяем user/system и assistant части
    user_msgs  = [m for m in msgs if m['role'] in ('system', 'user')]
    asst_msgs  = [m for m in msgs if m['role'] == 'assistant']

    # Длина prompt-части (только system + user)
    prompt_text = tokenizer.apply_chat_template(
        user_msgs,
        tokenize=False,
        add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']

    # Длина ответа
    if asst_msgs:
        answer_text = asst_msgs[-1]['content']
        answer_ids  = tokenizer(answer_text, add_special_tokens=False)['input_ids']
    else:
        answer_ids = []

    return len(prompt_ids), len(answer_ids)

In [ ]:
print("\nТокенизируем выборку (2000 примеров из verified)...")
SAMPLE_SIZE = min(2000, len(ds_verified))
sample = ds_verified.select(range(SAMPLE_SIZE))

prompt_lens  = []
answer_lens  = []
total_lens   = []
nums_counts  = []

for ex in sample:
    p_len, a_len = tokenize_example(ex, tokenizer)
    prompt_lens.append(p_len)
    answer_lens.append(a_len)
    total_lens.append(p_len + a_len)
    nums_counts.append(len(ex['nums']))

prompt_lens = np.array(prompt_lens)
answer_lens = np.array(answer_lens)
total_lens  = np.array(total_lens)
nums_counts = np.array(nums_counts)

In [ ]:
print(f"\n── Длина ПРОМПТА (system+user) ─────────────────────────────────")
for stat, val in [("min", prompt_lens.min()), ("max", prompt_lens.max()),
                  ("mean", prompt_lens.mean()), ("std", prompt_lens.std()),
                  ("p50", np.percentile(prompt_lens, 50)),
                  ("p90", np.percentile(prompt_lens, 90))]:
    print(f"  {stat:>5} = {val:>7.1f} токенов")

print(f"\n── Длина ОТВЕТА (assistant CoT + <answer>) ─────────────────────")
for stat, val in [("min", answer_lens.min()), ("max", answer_lens.max()),
                  ("mean", answer_lens.mean()), ("std", answer_lens.std()),
                  ("p50", np.percentile(answer_lens, 50)),
                  ("p75", np.percentile(answer_lens, 75)),
                  ("p90", np.percentile(answer_lens, 90)),
                  ("p95", np.percentile(answer_lens, 95)),
                  ("p99", np.percentile(answer_lens, 99))]:
    print(f"  {stat:>5} = {val:>7.1f} токенов")

print(f"\n── Длина ВСЕГО ПРИМЕРА (prompt + answer) ───────────────────────")
for stat, val in [("min", total_lens.min()), ("max", total_lens.max()),
                  ("mean", total_lens.mean()), ("std", total_lens.std()),
                  ("p50", np.percentile(total_lens, 50)),
                  ("p75", np.percentile(total_lens, 75)),
                  ("p90", np.percentile(total_lens, 90)),
                  ("p95", np.percentile(total_lens, 95)),
                  ("p99", np.percentile(total_lens, 99))]:
    print(f"  {stat:>5} = {val:>7.1f} токенов")

In [ ]:
# По группам сложности
print(f"\n── Длина ОТВЕТА по количеству чисел ────────────────────────────")
print(f"  {'nums':>6} | {'count':>6} | {'min':>6} | {'mean':>7} | {'p90':>7} | {'max':>7}")
print(f"  {'-'*6}-+-{'-'*6}-+-{'-'*6}-+-{'-'*7}-+-{'-'*7}-+-{'-'*7}")
for n in sorted(set(nums_counts)):
    mask = nums_counts == n
    a = answer_lens[mask]
    t = total_lens[mask]
    print(f"  {n:>6} | {mask.sum():>6} | {a.min():>6} | {a.mean():>7.0f} | "
          f"{np.percentile(a, 90):>7.0f} | {a.max():>7}")

# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 4: ПОКРЫТИЕ max_seq_length")
print("=" * 70)

print(f"\n{'max_seq':>10} | {'покрытие':>9} | {'обрезается':>11} | {'truncated_ans':>14} | {'рекомендация'}")
print("-" * 75)

recommended_max_seq = None
for max_seq in MAX_SEQ_OPTIONS:
    covered  = (total_lens <= max_seq).sum()
    truncated = (total_lens > max_seq).sum()
    # Сколько примеров, у которых ответ обрезается (prompt влезает, а ответ нет)
    ans_truncated = ((prompt_lens < max_seq) & (total_lens > max_seq)).sum()
    pct  = covered / len(total_lens) * 100
    rec  = ""
    if pct >= 95 and recommended_max_seq is None:
        rec = "← рекомендован (95%+)"
        recommended_max_seq = max_seq
    elif pct >= 90:
        rec = "← хорошо (90%+)"
    elif pct >= 75:
        rec = "← приемлемо"
    print(f"{max_seq:>10} | {pct:>8.1f}% | {truncated:>11} | {ans_truncated:>14} | {rec}")

if recommended_max_seq is None:
    recommended_max_seq = 2048
print(f"\n  Рекомендуемый max_seq_length = {recommended_max_seq}")
print(f"  (покрывает {(total_lens <= recommended_max_seq).mean()*100:.1f}% примеров)")

# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 5: АНАЛИЗ PACKING")
print("=" * 70)

print("\nПри packing несколько примеров упаковываются в одну последовательность.")
print("Ключевой вопрос: сколько примеров влезает в один пакет?\n")

for max_seq in [1024, 1536, 2048]:
    bins = defaultdict(int)
    for length in total_lens:
        n_fit = max(1, int(max_seq // length))
        bins[min(n_fit, 5)] += 1
    total = len(total_lens)
    print(f"  max_seq={max_seq}:")
    for k in sorted(bins):
        label = f"{k}+" if k == 5 else str(k)
        print(f"    {label} примеров в пакете: {bins[k]:>5} ({bins[k]/total*100:5.1f}%)")

    waste = sum(max_seq - (total_lens[i] % max_seq if total_lens[i] % max_seq != 0 else max_seq)
                for i in range(len(total_lens)))
    waste_pct = waste / (max_seq * len(total_lens)) * 100
    print(f"    Потери на паддинг БЕЗ packing: {waste_pct:.1f}% токенов\n")

# Вариативность длин (ключ к packing)
cv = total_lens.std() / total_lens.mean()
print(f"  Коэффициент вариации длин = {cv:.2f}")
if cv > 0.4:
    print(f"  ✓ Высокая вариативность → packing даст значительный выигрыш")
elif cv > 0.2:
    print(f"  ~ Умеренная вариативность → packing выгоден")
else:
    print(f"  ~ Низкая вариативность → packing менее критичен")

# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 6: СОВМЕСТИМОСТЬ TRAIN vs TEST")
print("=" * 70)

# Загружаем тестовый датасет из HF (10k split)
print("\nЗагружаем тестовый сплит (test) из датасета...")
try:
    ds_test_hf = load_dataset(DATASET_NAME, ALL_CONFIG, split="test")
    test_nums_dist = Counter(len(ex['nums']) for ex in ds_test_hf)
    print(f"  Тестовый сплит HF: {len(ds_test_hf)} примеров")
    hf_test_available = True
except Exception:
    hf_test_available = False
    print("  Тестовый сплит HF недоступен, используем только verified")

# Сравниваем распределения
train_total = sum(cnt_verified.values())
print(f"\n{'nums':>6} | {'verified (train)':>18} | {'all (train)':>13}", end="")
if hf_test_available:
    test_total = sum(test_nums_dist.values())
    print(f" | {'test (HF)':>12} | {'gap train-test':>14}")
    print("-" * 80)
    for n in sorted(set(list(cnt_verified.keys()) + list(test_nums_dist.keys()))):
        tr_v = cnt_verified.get(n, 0)
        tr_a = cnt_all.get(n, 0)
        te   = test_nums_dist.get(n, 0)
        tr_v_pct = tr_v / train_total * 100
        tr_a_pct = tr_a / len(ds_all) * 100
        te_pct   = te / test_total * 100
        gap      = tr_v_pct - te_pct
        flag     = " ⚠" if abs(gap) > 10 else ""
        print(f"  {n:>6} | {tr_v:>6} ({tr_v_pct:5.1f}%) | "
              f"{tr_a:>5} ({tr_a_pct:4.1f}%) | "
              f"{te:>5} ({te_pct:4.1f}%) | {gap:>+7.1f}%{flag}")
else:
    print(f" | {'gap':>12}")
    print("-" * 55)
    all_total = len(ds_all)
    for n in sorted(cnt_verified.keys()):
        tr_v = cnt_verified.get(n, 0)
        tr_a = cnt_all.get(n, 0)
        tr_v_pct = tr_v / train_total * 100
        tr_a_pct = tr_a / all_total * 100
        gap = tr_v_pct - tr_a_pct
        flag = " ⚠" if abs(gap) > 10 else ""
        print(f"  {n:>6} | {tr_v:>6} ({tr_v_pct:5.1f}%) | "
              f"{tr_a:>5} ({tr_a_pct:4.1f}%) | {gap:>+7.1f}%{flag}")

# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 7: КАЧЕСТВО ДАННЫХ")
print("=" * 70)

# Проверка дублей
targets_and_nums = [(ex['target'], tuple(sorted(ex['nums']))) for ex in ds_verified]
n_dupes = len(targets_and_nums) - len(set(targets_and_nums))
print(f"\n  Дубликатов (одинаковый target+nums): {n_dupes}")

# Проверка формата ответа
answer_stats = {
    'has_answer_tag': 0,
    'has_think_tag' : 0,
    'answer_correct': 0,   # уравнение математически верно
    'answer_wrong'  : 0,
    'answer_no_tag' : 0,
    'answer_lengths': [],
}

print(f"\n  Проверяем корректность уравнений в ответах (выборка 500)...")
checked = 0
for ex in ds_verified.select(range(min(500, len(ds_verified)))):
    msgs = ex['prompt']
    asst = next((m for m in reversed(msgs) if m['role'] == 'assistant'), None)
    if asst is None:
        answer_stats['answer_no_tag'] += 1
        continue
    content = asst['content']
    if '<think>' in content:
        answer_stats['has_think_tag'] += 1
    m = re.search(r'<answer>(.*?)</answer>', content, re.DOTALL)
    if m:
        answer_stats['has_answer_tag'] += 1
        eq = m.group(1).strip()
        answer_stats['answer_lengths'].append(len(eq))
        # Верифицируем математически
        try:
            nums_in_eq = [int(x) for x in re.findall(r'\d+', eq)]
            nums_avail = sorted(ex['nums'])
            nums_used  = sorted(nums_in_eq)
            # Упрощённая проверка: все числа из доступных
            all_valid = all(n in nums_avail for n in nums_in_eq)
            result    = eval(eq)
            if abs(result - ex['target']) < 1e-6 and all_valid:
                answer_stats['answer_correct'] += 1
            else:
                answer_stats['answer_wrong'] += 1
        except Exception:
            answer_stats['answer_wrong'] += 1
    else:
        answer_stats['answer_no_tag'] += 1
    checked += 1

print(f"  Проверено: {checked} примеров")
print(f"    <think> тег         : {answer_stats['has_think_tag']:>4} ({answer_stats['has_think_tag']/checked*100:.1f}%)")
print(f"    <answer> тег        : {answer_stats['has_answer_tag']:>4} ({answer_stats['has_answer_tag']/checked*100:.1f}%)")
print(f"    Уравнение верно     : {answer_stats['answer_correct']:>4} ({answer_stats['answer_correct']/checked*100:.1f}%)")
print(f"    Уравнение неверно   : {answer_stats['answer_wrong']:>4} ({answer_stats['answer_wrong']/checked*100:.1f}%)")
print(f"    Нет тега / пусто    : {answer_stats['answer_no_tag']:>4} ({answer_stats['answer_no_tag']/checked*100:.1f}%)")

if answer_stats['answer_lengths']:
    eq_lens = answer_stats['answer_lengths']
    print(f"\n  Длина уравнений (символы, без CoT):")
    print(f"    min  = {min(eq_lens)}")
    print(f"    max  = {max(eq_lens)}")
    print(f"    mean = {np.mean(eq_lens):.1f}")
    print(f"\n  Примеры уравнений:")
    for ex in ds_verified.select(range(5)):
        msgs = ex['prompt']
        asst = next((m for m in reversed(msgs) if m['role'] == 'assistant'), None)
        if asst:
            m = re.search(r'<answer>(.*?)</answer>', asst['content'], re.DOTALL)
            if m:
                eq = m.group(1).strip()
                print(f"    nums={ex['nums']}, target={ex['target']}")
                print(f"    → {eq}")
                print()

# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 8: ВИЗУАЛИЗАЦИЯ")
print("=" * 70)

fig = plt.figure(figsize=(18, 14))
fig.suptitle("EDA: Countdown Distillation Dataset", fontsize=16, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── График 1: Распределение длин ответов ─────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.hist(answer_lens, bins=60, color='steelblue', alpha=0.8, edgecolor='white', linewidth=0.3)
for max_seq, color in [(1024, 'red'), (1536, 'orange'), (2048, 'green')]:
    prompt_mean = prompt_lens.mean()
    ax1.axvline(max_seq - prompt_mean, color=color, linestyle='--', linewidth=1.5,
                label=f'max_seq={max_seq} (ответ≤{int(max_seq-prompt_mean)})')
ax1.set_xlabel("Длина ответа (токены)", fontsize=11)
ax1.set_ylabel("Количество примеров", fontsize=11)
ax1.set_title("Распределение длин ОТВЕТА (assistant)", fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# ── График 2: Распределение длин ответов по сложности ────────────────
ax2 = fig.add_subplot(gs[0, 2])
colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
for i, n in enumerate(sorted(set(nums_counts))):
    mask = nums_counts == n
    ax2.hist(answer_lens[mask], bins=30, alpha=0.6, color=colors[i], label=f'{n} чисел')
ax2.set_xlabel("Длина ответа (токены)", fontsize=10)
ax2.set_ylabel("Кол-во примеров", fontsize=10)
ax2.set_title("Длина ответа по сложности", fontsize=11, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

# ── График 3: Покрытие max_seq_length ────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
coverage = [(ms, (total_lens <= ms).mean() * 100) for ms in MAX_SEQ_OPTIONS]
ms_vals, cov_vals = zip(*coverage)
bars = ax3.bar(range(len(ms_vals)), cov_vals, color=['#f44336','#ff9800','#4caf50','#2196f3','#9c27b0'])
ax3.axhline(95, color='gray', linestyle='--', linewidth=1, label='95% порог')
ax3.axhline(90, color='gray', linestyle=':', linewidth=1, label='90% порог')
ax3.set_xticks(range(len(ms_vals)))
ax3.set_xticklabels([str(ms) for ms in ms_vals], fontsize=9)
ax3.set_xlabel("max_seq_length", fontsize=10)
ax3.set_ylabel("Покрытие (%)", fontsize=10)
ax3.set_title("Покрытие при разных\nmax_seq_length", fontsize=11, fontweight='bold')
ax3.set_ylim(50, 102)
ax3.legend(fontsize=8)
for bar, val in zip(bars, cov_vals):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

# ── График 4: Длина ответа (box plot по кол-ву чисел) ────────────────
ax4 = fig.add_subplot(gs[1, 1])
data_by_n = [answer_lens[nums_counts == n] for n in sorted(set(nums_counts))]
bp = ax4.boxplot(data_by_n, labels=[f'{n} чисел' for n in sorted(set(nums_counts))],
                 patch_artist=True, medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax4.set_xlabel("Кол-во чисел в задаче", fontsize=10)
ax4.set_ylabel("Длина ответа (токены)", fontsize=10)
ax4.set_title("Box plot: длина ответа\nпо сложности", fontsize=11, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# ── График 5: Распределение кол-ва чисел в verified vs all ───────────
ax5 = fig.add_subplot(gs[1, 2])
ns = sorted(set(list(cnt_verified.keys()) + list(cnt_all.keys())))
x = np.arange(len(ns))
w = 0.35
v_vals = [cnt_verified.get(n, 0) / train_total * 100 for n in ns]
a_vals = [cnt_all.get(n, 0) / len(ds_all) * 100 for n in ns]
ax5.bar(x - w/2, v_vals, w, label='verified', color='steelblue', alpha=0.8)
ax5.bar(x + w/2, a_vals, w, label='all',      color='coral',     alpha=0.8)
ax5.set_xticks(x)
ax5.set_xticklabels([f'{n} чисел' for n in ns], fontsize=9)
ax5.set_ylabel("Доля (%)", fontsize=10)
ax5.set_title("Распределение сложности\nverified vs all", fontsize=11, fontweight='bold')
ax5.legend(fontsize=9)
ax5.grid(axis='y', alpha=0.3)

# ── График 6: Суммарная длина (scatter: nums vs total_len) ────────────
ax6 = fig.add_subplot(gs[2, :2])
for i, n in enumerate(sorted(set(nums_counts))):
    mask = nums_counts == n
    ax6.scatter(np.full(mask.sum(), n) + np.random.randn(mask.sum()) * 0.1,
                total_lens[mask], alpha=0.15, s=8, color=colors[i], label=f'{n} чисел')
for max_seq, color in [(1024, 'red'), (1536, 'orange'), (2048, 'green')]:
    ax6.axhline(max_seq, color=color, linestyle='--', linewidth=1.5, label=f'max_seq={max_seq}')
ax6.set_xlabel("Кол-во чисел в задаче", fontsize=10)
ax6.set_ylabel("Полная длина примера (токены)", fontsize=10)
ax6.set_title("Длина примера (prompt+answer) vs сложность задачи", fontsize=11, fontweight='bold')
ax6.legend(fontsize=8, ncol=3)
ax6.grid(alpha=0.2)

# ── График 7: Распределение target ───────────────────────────────────
ax7 = fig.add_subplot(gs[2, 2])
ax7.hist(targets_verified, bins=40, color='mediumpurple', alpha=0.8,
         edgecolor='white', linewidth=0.3)
ax7.set_xlabel("Значение target", fontsize=10)
ax7.set_ylabel("Кол-во примеров", fontsize=10)
ax7.set_title("Распределение target\n(verified)", fontsize=11, fontweight='bold')
ax7.grid(axis='y', alpha=0.3)

plt.savefig("/home/claude/countdown_eda.png", dpi=150, bbox_inches='tight',
            facecolor='white')
print("  График сохранён: countdown_eda.png")
plt.show()

# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 9: ИТОГОВЫЕ РЕКОМЕНДАЦИИ")
print("=" * 70)

# Вычисляем рекомендуемый max_seq
for ms in MAX_SEQ_OPTIONS:
    cov = (total_lens <= ms).mean() * 100
    if cov >= 90:
        rec_max_seq = ms
        rec_cov = cov
        break
else:
    rec_max_seq = MAX_SEQ_OPTIONS[-1]
    rec_cov = (total_lens <= rec_max_seq).mean() * 100

# Выгода packing
cv_val = total_lens.std() / total_lens.mean()
packing_recommended = cv_val > 0.25

print(f"""
┌─────────────────────────────────────────────────────────────────┐
│  КОНФИГУРАЦИЯ ДЛЯ ОБУЧЕНИЯ                                      │
├─────────────────────────────────────────────────────────────────┤
│                                                                   │
│  max_seq_length  = {rec_max_seq:<6}  (покрывает {rec_cov:.1f}% примеров)           │
│  packing         = {str(packing_recommended):<6}  (CV длин = {cv_val:.2f}, {'высокая' if cv_val > 0.4 else 'умеренная'} вариативность)     │
│                                                                   │
│  Длина промпта   = {int(prompt_lens.mean()):<6} токенов (стабильная)               │
│  Длина ответа    = {int(answer_lens.mean()):<6} токенов (среднее)                 │
│  Длина ответа p90= {int(np.percentile(answer_lens, 90)):<6} токенов                                │
│                                                                   │
├─────────────────────────────────────────────────────────────────┤
│  ПРЕДУПРЕЖДЕНИЯ                                                   │
├─────────────────────────────────────────────────────────────────┤""")

# Предупреждения
warnings_list = []

n5_6_verified = (cnt_verified.get(5, 0) + cnt_verified.get(6, 0)) / train_total * 100
n5_6_all = (cnt_all.get(5, 0) + cnt_all.get(6, 0)) / len(ds_all) * 100
if n5_6_verified < 20:
    warnings_list.append(
        f"  ⚠ В verified только {n5_6_verified:.1f}% примеров с 5-6 числами!\n"
        f"    В тесте таких ~44%. Дополни данными из сплита 'all'."
    )

truncated_5_6 = ((nums_counts >= 5) & (total_lens > rec_max_seq)).sum()
total_5_6 = (nums_counts >= 5).sum()
if total_5_6 > 0 and truncated_5_6 / total_5_6 > 0.2:
    warnings_list.append(
        f"  ⚠ При max_seq={rec_max_seq}: {truncated_5_6}/{total_5_6} примеров с 5-6\n"
        f"    числами обрезаются. Рассмотри max_seq=2048 для лучшего качества."
    )

bad_answers = answer_stats['answer_wrong'] + answer_stats['answer_no_tag']
if bad_answers / checked > 0.05:
    warnings_list.append(
        f"  ⚠ {bad_answers/checked*100:.1f}% примеров с некорректным/отсутствующим\n"
        f"    ответом. Добавь фильтрацию перед обучением."
    )

if n_dupes > 0:
    warnings_list.append(f"  ⚠ Найдено {n_dupes} дубликатов — рекомендуется дедупликация.")

if not warnings_list:
    warnings_list.append("  ✓ Критических проблем не обнаружено.")

for w in warnings_list:
    print(f"│{w:<65}│")

print(f"""├─────────────────────────────────────────────────────────────────┤
│  ЧЕКЛИСТ ПЕРЕД ОБУЧЕНИЕМ                                         │
├─────────────────────────────────────────────────────────────────┤
│  [ ] Добавить примеры с 5-6 числами из сплита 'all'             │
│  [ ] Фильтровать примеры без <answer> тега                       │
│  [ ] Убедиться что все уравнения математически верны            │
│  [ ] Удалить дубликаты                                           │
│  [ ] Применить chat template Gemma (не Qwen!)                   │
│  [ ] Использовать train_on_responses_only                        │
│  [ ] Unsloth + packing=True для ускорения                       │
│  [ ] Проверить OOM при выбранном max_seq на T4                   │
└─────────────────────────────────────────────────────────────────┘
""")

# ─────────────────────────────────────────────────────────────────────
print("=" * 70)
print("  СЕКЦИЯ 10: ГЕНЕРАЦИЯ ФИНАЛЬНОГО КОНФИГА")
print("=" * 70)

config = {
    "model": {
        "student": "google/gemma-3-1b-it",
        "teacher": "Qwen/Qwen2.5-7B-Instruct",
        "load_in_4bit": True,
        "use_unsloth": True,
    },
    "dataset": {
        "name": DATASET_NAME,
        "config": DATASET_CONFIG,
        "train_size": len(ds_verified),
        "val_split": 0.05,
        "filter_no_answer": True,
        "deduplicate": True,
        "add_5_6_nums": n5_6_verified < 20,
    },
    "training": {
        "max_seq_length": rec_max_seq,
        "packing": packing_recommended,
        "lora_r": 16,
        "lora_alpha": 16,
        "lora_dropout": 0,
        "learning_rate": 2e-4,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "num_train_epochs": 2,
        "warmup_ratio": 0.05,
        "lr_scheduler_type": "cosine",
        "optim": "adamw_8bit",
        "bf16": False,
        "fp16": True,
        "max_grad_norm": 1.0,
        "use_gradient_checkpointing": "unsloth",
        "train_on_responses_only": True,
    },
    "evaluation": {
        "eval_strategy": "steps",
        "eval_steps": 200,
        "metric": "countdown_accuracy",
        "answer_tag": "<answer>",
        "float_tolerance": 1e-6,
    },
    "eda_stats": {
        "prompt_len_mean": int(prompt_lens.mean()),
        "answer_len_mean": int(answer_lens.mean()),
        "answer_len_p90": int(np.percentile(answer_lens, 90)),
        "answer_len_p99": int(np.percentile(answer_lens, 99)),
        "total_len_mean": int(total_lens.mean()),
        "coverage_at_rec_seq": float(f"{rec_cov:.1f}"),
        "pct_5_6_nums_verified": float(f"{n5_6_verified:.1f}"),
        "length_cv": float(f"{cv_val:.3f}"),
    }
}

print("\nКонфиг обучения:")
print(json.dumps(config, indent=2, ensure_ascii=False))

config_path = "/home/claude/training_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(f"\nКонфиг сохранён: {config_path}")

print("\n" + "=" * 70)
print("  АНАЛИЗ ЗАВЕРШЁН")
print("=" * 70)

In [ ]:
import re
import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer
 
warnings.filterwarnings("ignore")
 
# ── НАСТРОЙКИ ─────────────────────────────────────────────────────────
DATASET_NAME    = "HuggingFaceTB/Countdown-Task-GOLD"
VERIFIED_CONFIG = "verified_Qwen2.5-7B-Instruct"
ALL_CONFIG      = "all"
MODEL_NAME      = "google/gemma-3-1b-it"
MAX_SEQ_OPTIONS = [512, 768, 1024, 1536, 2048]
TOKENIZE_SAMPLE = 2000   # сколько примеров токенизировать (скорость vs точность)
VALIDATE_SAMPLE = 500    # сколько проверять математически
 
# ─────────────────────────────────────────────────────────────────────
print("=" * 70)
print("  СЕКЦИЯ 1: ЗАГРУЗКА ДАТАСЕТОВ")
print("=" * 70)
 
ds_verified = load_dataset(DATASET_NAME, VERIFIED_CONFIG, split="train")
ds_all      = load_dataset(DATASET_NAME, ALL_CONFIG,      split="train")
 
print(f"\nverified ({VERIFIED_CONFIG}): {len(ds_verified):>6} примеров")
print(f"all:                          {len(ds_all):>6} примеров")
print(f"Колонки: {ds_verified.column_names}")
 
# Показываем структуру одного примера
ex0 = ds_verified[0]
print(f"\n── Пример #0 ──────────────────────────────────────────────────")
print(f"  target  : {ex0['target']}")
print(f"  nums    : {ex0['nums']}")
print(f"  messages: {len(ex0['messages'])} сообщений")
for i, msg in enumerate(ex0['messages']):
    preview = msg['content'].replace('\n', ' ')[:100]
    print(f"    [{i}] role={msg['role']:9} | {preview}...")
 
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 2: БАЗОВАЯ СТАТИСТИКА")
print("=" * 70)
 
def dataset_stats(ds, name):
    nums_lens = [len(ex['nums']) for ex in ds]
    targets   = [ex['target']   for ex in ds]
    cnt = Counter(nums_lens)
    total = len(ds)
 
    print(f"\n── {name} ({total} примеров) ──────────────────────────────────")
    print(f"  Количество чисел (nums):")
    for k in sorted(cnt):
        print(f"    {k} чисел: {cnt[k]:>6}  ({cnt[k]/total*100:5.1f}%)")
 
    print(f"\n  Target:  min={min(targets)}  max={max(targets)}"
          f"  mean={np.mean(targets):.0f}"
          f"  p50={np.percentile(targets,50):.0f}"
          f"  p90={np.percentile(targets,90):.0f}")
    return cnt, targets
 
cnt_verified, tgt_verified = dataset_stats(ds_verified, VERIFIED_CONFIG)
cnt_all,      tgt_all      = dataset_stats(ds_all,      "all")
 
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 3: ТОКЕНИЗАЦИЯ")
print("=" * 70)
 
print(f"\nЗагружаем токенизатор {MODEL_NAME} ...")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"  vocab_size    = {tok.vocab_size:,}")
print(f"  chat_template = {'есть' if tok.chat_template else 'НЕТ!'}")
print(f"  bos_token     = {repr(tok.bos_token)}")
print(f"  eos_token     = {repr(tok.eos_token)}")
 
# Показываем как выглядит промпт после apply_chat_template
sample_user_msgs = [
    {"role": "system", "content": ex0['messages'][0]['content']},
    {"role": "user",   "content": ex0['messages'][1]['content']},
]
prompt_rendered = tok.apply_chat_template(
    sample_user_msgs, tokenize=False, add_generation_prompt=True
)
print(f"\n── Пример промпта после apply_chat_template ────────────────────")
print(prompt_rendered[:400])
print("...")
 
def tokenize_example(ex, tokenizer):
    """Возвращает (prompt_len, answer_len, total_len)."""
    msgs = ex['messages']
    # Промпт = system + user
    user_part = [m for m in msgs if m['role'] in ('system', 'user')]
    prompt_text = tokenizer.apply_chat_template(
        user_part, tokenize=False, add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
 
    # Ответ = content assistant
    asst = next((m for m in msgs if m['role'] == 'assistant'), None)
    if asst:
        asst_ids = tokenizer(asst['content'], add_special_tokens=False)['input_ids']
    else:
        asst_ids = []
 
    p = len(prompt_ids)
    a = len(asst_ids)
    return p, a, p + a
 
print(f"\nТокенизируем {TOKENIZE_SAMPLE} примеров из verified ...")
sample = ds_verified.select(range(min(TOKENIZE_SAMPLE, len(ds_verified))))
 
prompt_lens = []
answer_lens = []
total_lens  = []
nums_counts = []
 
for ex in sample:
    p, a, t = tokenize_example(ex, tok)
    prompt_lens.append(p)
    answer_lens.append(a)
    total_lens.append(t)
    nums_counts.append(len(ex['nums']))
 
prompt_lens = np.array(prompt_lens)
answer_lens = np.array(answer_lens)
total_lens  = np.array(total_lens)
nums_counts = np.array(nums_counts)
 
def print_stats(arr, label):
    print(f"\n── {label} ─────────────────────────────────────────────────")
    for name, val in [
        ("min",  arr.min()),  ("max",  arr.max()),
        ("mean", arr.mean()), ("std",  arr.std()),
        ("p50",  np.percentile(arr, 50)),
        ("p75",  np.percentile(arr, 75)),
        ("p90",  np.percentile(arr, 90)),
        ("p95",  np.percentile(arr, 95)),
        ("p99",  np.percentile(arr, 99)),
    ]:
        print(f"  {name:>5} = {val:>8.1f} токенов")
 
print_stats(prompt_lens, "ДЛИНА ПРОМПТА (system + user)")
print_stats(answer_lens, "ДЛИНА ОТВЕТА  (assistant)")
print_stats(total_lens,  "ПОЛНАЯ ДЛИНА  (prompt + answer)")
 
print(f"\n── Длина ответа по количеству чисел ────────────────────────────")
hdr = f"  {'nums':>5} | {'n':>5} | {'min':>5} | {'mean':>6} | {'p75':>6} | {'p90':>6} | {'p99':>6} | {'max':>6}"
print(hdr)
print("  " + "-" * (len(hdr) - 2))
for n in sorted(set(nums_counts)):
    mask = nums_counts == n
    a = answer_lens[mask]
    t = total_lens[mask]
    print(f"  {n:>5} | {mask.sum():>5} | {a.min():>5.0f} | {a.mean():>6.0f} | "
          f"{np.percentile(a,75):>6.0f} | {np.percentile(a,90):>6.0f} | "
          f"{np.percentile(a,99):>6.0f} | {a.max():>6.0f}")
 
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 4: ПОКРЫТИЕ max_seq_length")
print("=" * 70)
 
print(f"\n  {'max_seq':>8} | {'покрытие':>9} | {'обрезано':>9} | "
      f"{'из них 5-6 чисел':>17} | {'рекомендация'}")
print("  " + "-" * 72)
 
rec_max_seq = None
for ms in MAX_SEQ_OPTIONS:
    cov   = (total_lens <= ms).sum()
    trunc = (total_lens >  ms).sum()
    # Среди обрезанных — сколько сложных (5-6 чисел)
    trunc_hard = ((total_lens > ms) & (nums_counts >= 5)).sum()
    pct = cov / len(total_lens) * 100
    rec = ""
    if pct >= 95 and rec_max_seq is None:
        rec = "← рекомендован (95%+)"
        rec_max_seq = ms
    elif pct >= 90:
        rec = "← хорошо (90%+)"
    elif pct >= 75:
        rec = "← приемлемо"
    print(f"  {ms:>8} | {pct:>8.1f}% | {trunc:>9} | {trunc_hard:>17} | {rec}")
 
if rec_max_seq is None:
    rec_max_seq = MAX_SEQ_OPTIONS[-1]
 
rec_cov = (total_lens <= rec_max_seq).mean() * 100
print(f"\n  Рекомендуется: max_seq_length = {rec_max_seq}  "
      f"(покрывает {rec_cov:.1f}%)")
 
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 5: ВЫГОДА ОТ PACKING")
print("=" * 70)
 
cv = total_lens.std() / total_lens.mean()
print(f"\n  Коэффициент вариации длин CV = {cv:.3f}")
if   cv > 0.40: verdict = "Высокая — packing даст значительный выигрыш (рекомендован)"
elif cv > 0.20: verdict = "Умеренная — packing выгоден"
else:           verdict = "Низкая — packing менее критичен"
print(f"  Вариативность: {verdict}")
 
print(f"\n  Сколько примеров влезает в один пакет (без учёта паддинга):")
for ms in [1024, 1536, 2048]:
    fits = defaultdict(int)
    for ln in total_lens:
        n = max(1, int(ms // ln))
        fits[min(n, 5)] += 1
    total = len(total_lens)
    padding_waste = sum(
        ms - (ln % ms if ln % ms != 0 else ms)
        for ln in total_lens
    )
    waste_pct = padding_waste / (ms * total) * 100
    print(f"\n  max_seq={ms}  (потери паддинга без packing: {waste_pct:.1f}%):")
    for k in sorted(fits):
        lbl = f"{k}+" if k == 5 else str(k)
        print(f"    {lbl} примера в пакете: {fits[k]:>5} ({fits[k]/total*100:5.1f}%)")
 
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 6: СОВМЕСТИМОСТЬ TRAIN vs TEST")
print("=" * 70)
 
print("\nЗагружаем тестовый сплит HF ...")
try:
    ds_test = load_dataset(DATASET_NAME, ALL_CONFIG, split="test")
    cnt_test = Counter(len(ex['nums']) for ex in ds_test)
    test_total = len(ds_test)
    hf_test_ok = True
    print(f"  Тестовый сплит HF: {test_total} примеров")
except Exception as e:
    hf_test_ok = False
    print(f"  Недоступен: {e}")
 
ver_total = len(ds_verified)
all_total = len(ds_all)
ns = sorted(set(cnt_verified) | set(cnt_all))
 
if hf_test_ok:
    ns_all = sorted(set(ns) | set(cnt_test))
    print(f"\n  {'nums':>5} | {'verified %':>11} | {'all %':>7} | "
          f"{'test %':>7} | {'gap ver-test':>13}")
    print("  " + "-" * 55)
    for n in ns_all:
        v = cnt_verified.get(n, 0) / ver_total * 100
        a = cnt_all.get(n, 0)     / all_total  * 100
        t = cnt_test.get(n, 0)    / test_total * 100
        gap = v - t
        flag = "  ⚠" if abs(gap) > 10 else ""
        print(f"  {n:>5} | {v:>10.1f}% | {a:>6.1f}% | {t:>6.1f}% | {gap:>+10.1f}%{flag}")
else:
    print(f"\n  {'nums':>5} | {'verified %':>11} | {'all %':>7} | {'gap':>8}")
    print("  " + "-" * 40)
    for n in ns:
        v = cnt_verified.get(n, 0) / ver_total * 100
        a = cnt_all.get(n, 0)      / all_total  * 100
        flag = "  ⚠" if abs(v - a) > 10 else ""
        print(f"  {n:>5} | {v:>10.1f}% | {a:>6.1f}% | {v-a:>+7.1f}%{flag}")
 
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 7: КАЧЕСТВО ДАННЫХ")
print("=" * 70)
 
# Дубликаты
keys = [(ex['target'], tuple(sorted(ex['nums']))) for ex in ds_verified]
n_dupes = len(keys) - len(set(keys))
print(f"\n  Дубликатов (одинаковый target+nums): {n_dupes}")
 
# Статистика тегов и математическая корректность
n_check = min(VALIDATE_SAMPLE, len(ds_verified))
stats = dict(has_think=0, has_answer=0, correct=0, wrong=0, no_tag=0,
             no_asst=0, eval_error=0)
 
for ex in ds_verified.select(range(n_check)):
    msgs = ex['messages']
    asst = next((m for m in msgs if m['role'] == 'assistant'), None)
    if asst is None:
        stats['no_asst'] += 1
        continue
    c = asst['content']
    if '<think>'  in c: stats['has_think']  += 1
    if '<answer>' in c: stats['has_answer'] += 1
 
    m = re.search(r'<answer>(.*?)</answer>', c, re.DOTALL)
    if not m:
        stats['no_tag'] += 1
        continue
 
    raw = m.group(1).strip()
    # Формат: "expr = target" → берём только левую часть
    expr = raw.split('=')[0].strip() if '=' in raw else raw
    try:
        nums_in_expr = [int(x) for x in re.findall(r'\d+', expr)]
        nums_avail   = sorted(ex['nums'])
        # Все числа из выражения должны быть в доступных
        valid_nums = all(nums_in_expr.count(n) <= nums_avail.count(n)
                        for n in set(nums_in_expr))
        result = eval(expr)
        if abs(result - ex['target']) < 1e-6 and valid_nums:
            stats['correct'] += 1
        else:
            stats['wrong'] += 1
    except Exception:
        stats['eval_error'] += 1
        stats['wrong'] += 1
 
print(f"\n  Проверено {n_check} примеров:")
for key, val in stats.items():
    if val > 0:
        print(f"    {key:>15}: {val:>5}  ({val/n_check*100:5.1f}%)")
 
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 8: СТАТИСТИКА УРАВНЕНИЙ В <answer>")
print("=" * 70)
 
# Формат: всегда "expr = target" или иногда просто "expr"?
has_eq_sign    = 0
no_eq_sign     = 0
answer_char_lens = []
example_answers  = []
 
for ex in ds_verified.select(range(min(200, len(ds_verified)))):
    asst = next((m for m in ex['messages'] if m['role'] == 'assistant'), None)
    if asst is None: continue
    m = re.search(r'<answer>(.*?)</answer>', asst['content'], re.DOTALL)
    if not m: continue
    raw = m.group(1).strip()
    answer_char_lens.append(len(raw))
    if '=' in raw:
        has_eq_sign += 1
    else:
        no_eq_sign  += 1
    if len(example_answers) < 8:
        example_answers.append((ex['nums'], ex['target'], raw))
 
total_ans = has_eq_sign + no_eq_sign
print(f"\n  Из {total_ans} проверенных ответов:")
print(f"    Содержат '= target': {has_eq_sign} ({has_eq_sign/total_ans*100:.1f}%)")
print(f"    Без '=':             {no_eq_sign}  ({no_eq_sign/total_ans*100:.1f}%)")
 
print(f"\n  Длина строки внутри <answer> (символы):")
cl = np.array(answer_char_lens)
print(f"    min={cl.min()}  max={cl.max()}  mean={cl.mean():.1f}  p90={np.percentile(cl,90):.0f}")
 
print(f"\n  Примеры:")
for nums, target, raw in example_answers:
    print(f"    nums={nums}, target={target}")
    print(f"    → {raw}\n")
 
# ВАЖНЫЙ ВЫВОД ДЛЯ INFERENCE
print(f"  ⚠ ВАЖНО ДЛЯ INFERENCE:")
if has_eq_sign > 0:
    print(f"    Ответ содержит '= target' → для submission нужно split('=')[0].strip()")
    print(f"    Пример: '93 - (78 - 46) = 61'  →  '93 - (78 - 46)'")
else:
    print(f"    Ответ без '=' → используй содержимое <answer> напрямую")
 
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 9: ВИЗУАЛИЗАЦИЯ")
print("=" * 70)
 
fig = plt.figure(figsize=(18, 14))
fig.suptitle("EDA: Countdown Distillation Dataset\n"
             f"(verified={len(ds_verified)}, model={MODEL_NAME})",
             fontsize=14, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
COLORS = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
 
# 1 — Гистограмма длин ответов
ax1 = fig.add_subplot(gs[0, :2])
ax1.hist(answer_lens, bins=60, color='steelblue', alpha=0.85,
         edgecolor='white', linewidth=0.3)
p_mean = int(prompt_lens.mean())
for ms, col in [(1024,'red'), (1536,'orange'), (2048,'green')]:
    budget = ms - p_mean
    ax1.axvline(budget, color=col, ls='--', lw=1.8,
                label=f'max_seq={ms} (ответ≤{budget})')
ax1.set_xlabel("Длина ответа (токены)", fontsize=11)
ax1.set_ylabel("Кол-во примеров", fontsize=11)
ax1.set_title("Распределение длин ответа (assistant)", fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)
 
# 2 — Длины ответов по сложности
ax2 = fig.add_subplot(gs[0, 2])
for i, n in enumerate(sorted(set(nums_counts))):
    ax2.hist(answer_lens[nums_counts == n], bins=30, alpha=0.6,
             color=COLORS[i % len(COLORS)], label=f'{n} чисел')
ax2.set_xlabel("Длина ответа (токены)", fontsize=10)
ax2.set_title("Длина ответа\nпо сложности", fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)
 
# 3 — Покрытие max_seq
ax3 = fig.add_subplot(gs[1, 0])
covs = [(total_lens <= ms).mean() * 100 for ms in MAX_SEQ_OPTIONS]
bars = ax3.bar(range(len(MAX_SEQ_OPTIONS)), covs,
               color=['#f44336','#ff9800','#4caf50','#2196f3','#9c27b0'])
ax3.axhline(95, color='gray', ls='--', lw=1.2, label='95%')
ax3.axhline(90, color='gray', ls=':',  lw=1.2, label='90%')
ax3.set_xticks(range(len(MAX_SEQ_OPTIONS)))
ax3.set_xticklabels([str(ms) for ms in MAX_SEQ_OPTIONS], fontsize=9)
ax3.set_xlabel("max_seq_length", fontsize=10)
ax3.set_ylabel("Покрытие (%)", fontsize=10)
ax3.set_title("Покрытие при разных\nmax_seq_length", fontsize=11, fontweight='bold')
ax3.set_ylim(40, 103)
ax3.legend(fontsize=8)
for bar, v in zip(bars, covs):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{v:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)
 
# 4 — Box plot длин ответов
ax4 = fig.add_subplot(gs[1, 1])
box_data = [answer_lens[nums_counts == n] for n in sorted(set(nums_counts))]
bp = ax4.boxplot(box_data,
                 labels=[f'{n} чис.' for n in sorted(set(nums_counts))],
                 patch_artist=True,
                 medianprops=dict(color='white', linewidth=2))
for patch, col in zip(bp['boxes'], COLORS):
    patch.set_facecolor(col); patch.set_alpha(0.75)
ax4.set_xlabel("Кол-во чисел", fontsize=10)
ax4.set_ylabel("Длина ответа (токены)", fontsize=10)
ax4.set_title("Box plot: длина ответа\nпо сложности", fontsize=11, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)
 
# 5 — Распределение nums: verified vs all
ax5 = fig.add_subplot(gs[1, 2])
ns_plot = sorted(set(cnt_verified) | set(cnt_all))
x = np.arange(len(ns_plot))
w = 0.35
v_pct = [cnt_verified.get(n,0)/ver_total*100 for n in ns_plot]
a_pct = [cnt_all.get(n,0)/all_total*100       for n in ns_plot]
ax5.bar(x-w/2, v_pct, w, label='verified', color='steelblue', alpha=0.85)
ax5.bar(x+w/2, a_pct, w, label='all',      color='coral',     alpha=0.85)
ax5.set_xticks(x)
ax5.set_xticklabels([f'{n} чис.' for n in ns_plot], fontsize=9)
ax5.set_ylabel("Доля (%)", fontsize=10)
ax5.set_title("Распределение сложности\nverified vs all", fontsize=11, fontweight='bold')
ax5.legend(fontsize=9)
ax5.grid(axis='y', alpha=0.3)
 
# 6 — Scatter: nums_count vs total_len
ax6 = fig.add_subplot(gs[2, :2])
for i, n in enumerate(sorted(set(nums_counts))):
    mask = nums_counts == n
    jitter = np.random.randn(mask.sum()) * 0.07
    ax6.scatter(np.full(mask.sum(), n) + jitter, total_lens[mask],
                alpha=0.12, s=7, color=COLORS[i%len(COLORS)], label=f'{n} чисел')
for ms, col in [(1024,'red'), (1536,'orange'), (2048,'green')]:
    ax6.axhline(ms, color=col, ls='--', lw=1.5, label=f'max_seq={ms}')
ax6.set_xlabel("Кол-во чисел в задаче", fontsize=10)
ax6.set_ylabel("Полная длина (токены)", fontsize=10)
ax6.set_title("Длина примера (prompt+answer) vs сложность", fontsize=11, fontweight='bold')
ax6.legend(fontsize=8, ncol=3)
ax6.grid(alpha=0.2)
 
# 7 — Распределение target
ax7 = fig.add_subplot(gs[2, 2])
ax7.hist(tgt_verified, bins=40, color='mediumpurple', alpha=0.85,
         edgecolor='white', linewidth=0.3)
ax7.set_xlabel("Значение target", fontsize=10)
ax7.set_ylabel("Кол-во примеров", fontsize=10)
ax7.set_title("Распределение target (verified)", fontsize=11, fontweight='bold')
ax7.grid(axis='y', alpha=0.3)
 
plt.savefig("countdown_eda.png", dpi=150, bbox_inches='tight', facecolor='white')
print("  График сохранён: countdown_eda.png")
plt.show()
 
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  СЕКЦИЯ 10: ИТОГОВЫЕ РЕКОМЕНДАЦИИ")
print("=" * 70)
 
n5_6_ver = (cnt_verified.get(5,0) + cnt_verified.get(6,0)) / ver_total * 100
n5_6_all = (cnt_all.get(5,0)      + cnt_all.get(6,0))      / all_total * 100
trunc_at_rec = (total_lens > rec_max_seq).sum()
trunc_hard   = ((total_lens > rec_max_seq) & (nums_counts >= 5)).sum()
need_aug     = n5_6_ver < 20
 
warnings_out = []
if need_aug:
    warnings_out.append(
        f"⚠  verified содержит только {n5_6_ver:.1f}% задач с 5-6 числами.\n"
        f"   В тесте таких ~44%. Добавь примеры из сплита 'all'."
    )
if trunc_at_rec > 0:
    warnings_out.append(
        f"⚠  При max_seq={rec_max_seq}: {trunc_at_rec} примеров обрезается\n"
        f"   (из них {trunc_hard} с 5-6 числами). Рассмотри max_seq=2048."
    )
if stats['no_tag'] + stats['no_asst'] > 0:
    bad = stats['no_tag'] + stats['no_asst']
    warnings_out.append(
        f"⚠  {bad} примеров без тега <answer> — отфильтруй перед обучением."
    )
if n_dupes > 0:
    warnings_out.append(f"⚠  Найдено {n_dupes} дубликатов — дедуплицируй.")
if not warnings_out:
    warnings_out.append("✓  Критических проблем не обнаружено.")
 
sep = "─" * 68
print(f"""
┌{sep}┐
│  КОНФИГУРАЦИЯ                                                      │
├{sep}┤
│  max_seq_length     = {rec_max_seq:<6}  (покрытие {rec_cov:.1f}%)                    │
│  packing            = {str(cv > 0.25):<6}  (CV длин = {cv:.3f})                     │
│  train_on_responses = True   (маскируем промпт)                    │
│                                                                    │
│  Промпт             ≈ {int(prompt_lens.mean()):<6} токенов (стабильный)              │
│  Ответ mean         = {int(answer_lens.mean()):<6} токенов                          │
│  Ответ p90          = {int(np.percentile(answer_lens,90)):<6} токенов                          │
│  Ответ p99          = {int(np.percentile(answer_lens,99)):<6} токенов                          │
├{sep}┤
│  ПАРСИНГ ДЛЯ INFERENCE                                             │
│  pattern = r'<answer>(.*?)</answer>'                               │
│  expr    = match.group(1).strip().split('=')[0].strip()            │
│  # '93 - (78 - 46) = 61'  →  '93 - (78 - 46)'                    │
├{sep}┤
│  ПРЕДУПРЕЖДЕНИЯ                                                    │""")
for w in warnings_out:
    for line in w.split('\n'):
        print(f"│  {line:<66}│")
print(f"└{sep}┘")
 
# Генерируем конфиг
config = {
    "model": {
        "student": MODEL_NAME,
        "teacher": "Qwen/Qwen2.5-7B-Instruct",
        "load_in_4bit": True,
        "use_unsloth": True,
        "attn_implementation": "kernels-community/flash-attn2",
    },
    "dataset": {
        "name": DATASET_NAME,
        "config": VERIFIED_CONFIG,
        "messages_column": "messages",   # ← важно: не 'prompt'
        "train_size": len(ds_verified),
        "val_split": 0.05,
        "filter_no_answer_tag": True,
        "deduplicate": n_dupes > 0,
        "augment_with_all_split": need_aug,
        "augment_reason": f"only {n5_6_ver:.1f}% examples with 5-6 nums in verified",
    },
    "answer_parsing": {
        "tag_pattern": r"<answer>(.*?)</answer>",
        "extract_expr": "split('=')[0].strip()",
        "note": "answer contains '= target', need to strip it for submission",
    },
    "training": {
        "max_seq_length": rec_max_seq,
        "packing": bool(cv > 0.25),
        "train_on_responses_only": True,
        "lora_r": 16,
        "lora_alpha": 16,
        "lora_dropout": 0,
        "target_modules": "all-linear",
        "learning_rate": 2e-4,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "num_train_epochs": 2,
        "warmup_ratio": 0.05,
        "lr_scheduler_type": "cosine",
        "optim": "adamw_8bit",
        "fp16": True,
        "bf16": False,
        "max_grad_norm": 1.0,
        "use_gradient_checkpointing": "unsloth",
        "dataset_num_proc": 2,
    },
    "evaluation": {
        "eval_strategy": "steps",
        "eval_steps": 200,
        "metric": "countdown_accuracy",
        "float_tolerance": 1e-6,
    },
    "eda_stats": {
        "prompt_len_mean":  int(prompt_lens.mean()),
        "answer_len_mean":  int(answer_lens.mean()),
        "answer_len_p90":   int(np.percentile(answer_lens, 90)),
        "answer_len_p99":   int(np.percentile(answer_lens, 99)),
        "total_len_mean":   int(total_lens.mean()),
        "total_len_p95":    int(np.percentile(total_lens, 95)),
        "rec_max_seq":      rec_max_seq,
        "coverage_pct":     round(rec_cov, 1),
        "length_cv":        round(float(cv), 3),
        "pct_5_6_nums_verified": round(n5_6_ver, 1),
        "pct_5_6_nums_all":      round(n5_6_all, 1),
        "answer_has_eq_sign":    True,
    }
}
 
with open("training_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
 
print("\nКонфиг сохранён: training_config.json")
print(json.dumps(config, indent=2, ensure_ascii=False))
 
print("\n" + "=" * 70)
print("  АНАЛИЗ ЗАВЕРШЁН")
print("=" * 70)